In [38]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
import gradio as gr


requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """ Return the title and contents of the website at the given url"""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    title = soup.title.string if soup.title else "No title found"
    for irrelevant in soup.body(["script", "style", "img", "input"]):
        irrelevant.decompose()
    text = soup.body.get_text(separator="\n", strip=True) if soup.body else "No body content found"
    return title + "\n\n" + text

In [39]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [40]:
def stream_ollama(prompt, model_name=None):
    print("Calling Ollama")
    try:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt},
        ]

        stream = ollama.chat.completions.create(
            model=model_name or os.getenv("OLLAMA_MODEL", "gemma3:270m"),
            messages=messages,
            stream=True,
        )

        result = ""
        for chunk in stream:
            text = chunk.choices[0].delta.content or ""
            result += text
            print("Chunk:", text)
            yield result
    except Exception as e:
        print("Ollama error:", e)
        yield f"Error: {e}"

In [41]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),
# Please take a look at the Intermediate Python guide in the guides folder

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [34]:
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [42]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if  model=="Ollama":
        result = stream_ollama(prompt)
    elif model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["Ollama","GPT", "Claude"], label="Select model", value="Ollama")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Claude"],
            ["Edward Donner", "https://edwarddonner.com", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Calling Ollama
Chunk: Here
Chunk: '
Chunk: s
Chunk:  a
Chunk:  short
Chunk: ,
Chunk:  professional
Chunk:  brochure
Chunk:  for
Chunk:  Edward
Chunk:  Donner
Chunk: ,
Chunk:  focusing
Chunk:  on
Chunk:  his
Chunk:  expertise
Chunk:  in
Chunk:  LL
Chunk: Ms
Chunk:  and
Chunk:  the
Chunk:  company
Chunk: '
Chunk: s
Chunk:  offerings
Chunk: :
Chunk: 


Chunk: **
Chunk: Edward
Chunk:  Donner
Chunk: :
Chunk:  AI
Chunk:  Expert
Chunk:  &
Chunk:  Commercial
Chunk:  Solution
Chunk:  Provider
Chunk: **
Chunk: 


Chunk: Discover
Chunk:  how
Chunk:  Edward
Chunk:  Donner
Chunk:  can
Chunk:  help
Chunk:  businesses
Chunk:  leverage
Chunk:  the
Chunk:  power
Chunk:  of
Chunk:  artificial
Chunk:  intelligence
Chunk:  to
Chunk:  drive
Chunk:  innovation
Chunk:  and
Chunk:  growth
Chunk: .
Chunk: 


Chunk: At
Chunk:  Edward
Chunk:  Donner
Chunk: ,
Chunk:  we
Chunk:  are
Chunk:  passionate
Chunk:  about
Chunk:  providing
Chunk:  cutting
Chunk: -
Chunk: edge
Chunk:  AI
Chunk:  solutions
Chunk:  tailored